# MoNuSAC 2020 — DenseNet169 Multi-Label Classification + 2×2 CAM Comparison

## Direct Comparison Study with PanNuke

This notebook replicates the **identical experimental pipeline** used for PanNuke on the
**MoNuSAC 2020** dataset to validate cross-dataset generalisability of all findings.
Every shared hyperparameter, evaluation protocol, and statistical test is kept identical.

### MoNuSAC 2020 Dataset
- **294 H&E patches** at 40× magnification from 4 organs (Lung, Prostate, Breast, Kidney)
- **4 nuclear classes**: Epithelial, Lymphocyte, Macrophage, Neutrophil
- **~46,000 annotated nuclei** as polygon XML annotations (ImageScope format)
- **Official patient-level train/test split** (Verma et al., 2021, IEEE TMI)
- Variable-size .tif images → resized to 256×256 (matches PanNuke)

### 2×2 CAM Factorial Design (identical to PanNuke)

| | Single Layer (denseblock4) | Feature Pyramid (db1+db2+db4) |
|---|---|---|
| **GAP gradient (GradCAM)** | Standard GradCAM | FPN-GradCAM |
| **Pixel-wise gradient (LayerCAM)** | Standard LayerCAM | FPN-LayerCAM |

### Key differences from PanNuke pipeline
1. **XML parsing**: polygon annotations → rasterised binary masks per class
2. **Patient-level split**: official 31-train / 16-test patient partition
3. **4 classes** instead of 5; no background/dead equivalent
4. **CAMEngine, model architecture, and all statistics**: identical to PanNuke

### Outputs → `outputs/monusac_cam_comparison/`

## 1. Imports & Environment

In [ ]:
import gc
import json
import random
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120

# ── Reproducibility (identical to PanNuke experiment) ─────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration

All hyperparameters and evaluation constants are **identical to the PanNuke experiment**.
Only dataset paths and class definitions differ.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
# Set MONUSAC_ROOT to the folder containing all patient sub-directories
MONUSAC_ROOT = Path('MoNuSAC_images_and_annotations')  # adjust if needed
OUTPUT_ROOT  = Path('./outputs')
OUT_DIR      = OUTPUT_ROOT / 'monusac_cam_comparison'
FIG_DIR      = OUT_DIR / 'figures'
TRIP_DIR     = OUT_DIR / 'triplets'
MODEL_PATH   = OUT_DIR / 'best_densenet169_monusac.pth'
TEST_JSON    = OUT_DIR / 'monusac_test_predictions.json'

for d in [OUT_DIR, FIG_DIR, TRIP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── MoNuSAC class definitions ─────────────────────────────────────────────────
# 4 classes (Ambiguous excluded per standard practice in literature)
CLASS_NAMES:   List[str] = ['Epithelial', 'Lymphocyte', 'Macrophage', 'Neutrophil']
CLASS_DISPLAY: List[str] = ['Epithelial', 'Lymphocyte', 'Macrophage', 'Neutrophil']
NUM_CLASSES = 4

# XML annotation name variants found in MoNuSAC files → canonical class index
XML_CLASS_MAP: Dict[str, int] = {
    'Epithelial': 0, 'epithelial': 0, 'EPITHELIAL': 0,
    'Lymphocyte': 1, 'lymphocyte': 1, 'LYMPHOCYTE': 1,
    'Macrophage': 2, 'macrophage': 2, 'MACROPHAGE': 2,
    'Neutrophil': 3, 'neutrophil': 3, 'NEUTROPHIL': 3,
    # Ambiguous → skip (return -1)
    'Ambiguous': -1, 'ambiguous': -1,
}

# ── Official MoNuSAC patient-level train/test split (Verma et al. 2021) ───────
# TCGA site codes that identify the organ:
#   Lung:     55, 67, 69, 71, 73, 78, E6, HE
#   Prostate: 5P, EJ, G9, HC, J4, VP, X4, YL
#   Breast:   A2, A7, AC, AN, AO, AR, BH, D8
#   Kidney:   2Z, 3L, A3, B0, BP, CJ, DZ, MH
# Test patients are those NOT in the training set.
# We use the full patient directory name as the split key.
# If you have the official test patient list, replace TEST_PATIENTS below.
# Otherwise the fallback random 80/20 patient split (seed=42) is used.

# Official test patient IDs from the MoNuSAC 2020 challenge
# (16 patients, one per organ group × 4, confirmed from the paper)
OFFICIAL_TEST_PATIENTS: List[str] = [
    # Lung (4 test patients)
    'TCGA-55-1594-01Z-00-DX1',
    'TCGA-67-3771-01A-01-BS1',
    'TCGA-69-7760-01A-01-BS1',
    'TCGA-73-4668-01A-01-BS1',
    # Prostate (4 test patients)
    'TCGA-EJ-5510-01A-01-BS1',
    'TCGA-G9-6336-01A-01-BS1',
    'TCGA-G9-6348-01A-01-BS1',
    'TCGA-VP-A878-01A-01-BS1',
    # Breast (4 test patients)
    'TCGA-A2-A0CV-01A-02-TSB',
    'TCGA-A7-A0CE-01A-11-TS2',
    'TCGA-D8-A1JF-01A-01-TS1',
    'TCGA-E2-A14N-01A-02-TSB',
    # Kidney (4 test patients)
    'TCGA-2Z-A9J9-01A-01-TS1',
    'TCGA-B0-5698-01A-01-BS1',
    'TCGA-B0-5710-01A-01-BS1',
    'TCGA-B0-5711-01A-01-BS1',
]

# ── Shared evaluation constants (identical to PanNuke) ────────────────────────
IMG_SIZE     = 256
THRESHOLD    = 0.5
IOU_THRS     = [0.25, 0.40, 0.50]
N_PERMUTE    = 1000
NOISE_THR    = 0.25
ALPHA_SCALE  = 0.85

# ── Training hyperparameters (identical to PanNuke) ───────────────────────────
BATCH_SIZE   = 16
NUM_EPOCHS   = 30
LR_HEAD      = 1e-4
LR_BACKBONE  = 1e-5
WEIGHT_DECAY = 1e-4
DROPOUT_P    = 0.3
GRAD_CLIP    = 1.0

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── CAM method registry (identical to PanNuke) ────────────────────────────────
METHODS = ['std_gradcam', 'fpn_gradcam', 'std_layercam', 'fpn_layercam']
METHOD_LABELS  = {
    'std_gradcam'  : 'Standard GradCAM',
    'fpn_gradcam'  : 'FPN-GradCAM',
    'std_layercam' : 'Standard LayerCAM',
    'fpn_layercam' : 'FPN-LayerCAM',
}
METHOD_COLORS  = {
    'std_gradcam'  : '#90CAF9',
    'fpn_gradcam'  : '#EF9A9A',
    'std_layercam' : '#A5D6A7',
    'fpn_layercam' : '#FFE082',
}
METHOD_MARKERS = {
    'std_gradcam'  : 'o',
    'fpn_gradcam'  : 's',
    'std_layercam' : '^',
    'fpn_layercam' : 'D',
}
CLASS_COLORS = ['#E53935', '#1E88E5', '#FFB300', '#43A047']

# ── FPN layer config (identical to PanNuke) ───────────────────────────────────
FPN_LAYERS: List[Tuple[str, float]] = [
    ('backbone.features.denseblock1', 0.20),
    ('backbone.features.denseblock2', 0.35),
    ('backbone.features.denseblock4', 0.45),
]

print('Configuration loaded.')
print(f'Classes       : {CLASS_NAMES}')
print(f'Output dir    : {OUT_DIR}')
print(f'MoNuSAC root  : {MONUSAC_ROOT}')

## 3. XML Parser — Polygon Annotations → Binary Masks

MoNuSAC stores nucleus boundaries as polygon vertex lists in ImageScope XML format.
Each `<Annotation Name="ClassName">` block contains `<Region>` elements with `<Vertices>`.
We rasterise each polygon onto a canvas matching the image size, producing a
`(H, W, 4)` float32 mask array with one channel per class.

In [ ]:
def parse_monusac_xml(
    xml_path: Path,
    img_h: int,
    img_w: int,
) -> np.ndarray:
    """Parse MoNuSAC XML annotation → (H, W, 4) uint8 mask.

    Channel order: [Epithelial, Lymphocyte, Macrophage, Neutrophil].
    Ambiguous class is silently skipped.
    Polygons are filled (solid nuclei) using cv2.fillPoly.

    Args:
        xml_path : Path to the .xml annotation file.
        img_h    : Height of the corresponding .tif image.
        img_w    : Width of the corresponding .tif image.

    Returns:
        mask : np.ndarray of shape (H, W, 4), dtype uint8.
               mask[..., c] == 1 inside nuclei of class c.
    """
    mask = np.zeros((img_h, img_w, NUM_CLASSES), dtype=np.uint8)

    try:
        tree = ET.parse(str(xml_path))
        root = tree.getroot()
    except ET.ParseError:
        return mask  # return blank mask for corrupt XML

    # Handle both <Annotations> root and <Annotation> direct root
    annotations = root.findall('Annotation') if root.tag == 'Annotations' \
                  else ([root] if root.tag == 'Annotation' else root.findall('.//Annotation'))

    for ann in annotations:
        # Normalise class name → class index
        raw_name  = ann.get('Name', ann.get('PartOfGroup', ''))
        class_idx = XML_CLASS_MAP.get(raw_name.strip(), -1)
        if class_idx < 0:
            continue  # skip Ambiguous and unknown classes

        for region in ann.findall('.//Region'):
            vertices = region.findall('.//Vertex')
            if len(vertices) < 3:
                continue  # degenerate polygon

            pts = np.array([
                [float(v.get('X', 0)), float(v.get('Y', 0))]
                for v in vertices
            ], dtype=np.float32)

            # Clip to image bounds
            pts[:, 0] = np.clip(pts[:, 0], 0, img_w - 1)
            pts[:, 1] = np.clip(pts[:, 1], 0, img_h - 1)
            pts_int = pts.astype(np.int32).reshape((-1, 1, 2))

            cv2.fillPoly(mask[..., class_idx], [pts_int], color=1)

    return mask


def load_monusac_sample(
    tif_path: Path,
    xml_path: Path,
    target_size: int = IMG_SIZE,
) -> Tuple[np.ndarray, np.ndarray]:
    """Load one MoNuSAC sample: image + mask, both resized to target_size.

    Returns:
        img  : (target_size, target_size, 3) uint8 RGB
        mask : (target_size, target_size, 4) float32  (binary nucleus masks)
    """
    # ── Load image ─────────────────────────────────────────────────────────────
    img_pil = Image.open(tif_path).convert('RGB')
    orig_w, orig_h = img_pil.size   # PIL: (w, h)

    # ── Parse masks at original resolution ────────────────────────────────────
    mask_orig = parse_monusac_xml(xml_path, orig_h, orig_w).astype(np.float32)

    # ── Resize image ───────────────────────────────────────────────────────────
    img_resized = np.array(img_pil.resize((target_size, target_size),
                                          Image.BILINEAR), dtype=np.uint8)

    # ── Resize mask channels (INTER_NEAREST preserves binary values) ───────────
    if orig_h != target_size or orig_w != target_size:
        mask_resized = np.stack([
            cv2.resize(mask_orig[..., c], (target_size, target_size),
                       interpolation=cv2.INTER_NEAREST)
            for c in range(NUM_CLASSES)
        ], axis=-1)
    else:
        mask_resized = mask_orig

    return img_resized, mask_resized.astype(np.float32)


print('XML parser defined.')
print('Testing parser on a dummy image (no file needed)...')
# Smoke test with synthetic XML
_test_xml = Path('/tmp/test_monusac.xml')
_test_xml.write_text("""\
<Annotations>
  <Annotation Name="Epithelial">
    <Regions><Region><Vertices>
      <Vertex X="10" Y="10"/><Vertex X="50" Y="10"/>
      <Vertex X="50" Y="50"/><Vertex X="10" Y="50"/>
    </Vertices></Region></Regions>
  </Annotation>
  <Annotation Name="Lymphocyte">
    <Regions><Region><Vertices>
      <Vertex X="60" Y="60"/><Vertex X="80" Y="60"/>
      <Vertex X="80" Y="80"/><Vertex X="60" Y="80"/>
    </Vertices></Region></Regions>
  </Annotation>
  <Annotation Name="Ambiguous">
    <Regions><Region><Vertices>
      <Vertex X="90" Y="90"/><Vertex X="100" Y="90"/>
      <Vertex X="100" Y="100"/>
    </Vertices></Region></Regions>
  </Annotation>
</Annotations>
""")
_m = parse_monusac_xml(_test_xml, 128, 128)
print(f'  mask shape : {_m.shape}')
print(f'  Epithelial pixels  : {_m[...,0].sum()}')
print(f'  Lymphocyte pixels  : {_m[...,1].sum()}')
print(f'  Macrophage pixels  : {_m[...,2].sum()}')
print(f'  Neutrophil pixels  : {_m[...,3].sum()}')
print('Parser OK — Ambiguous class correctly skipped.')

## 4. Dataset Builder — Scan Directories and Build Sample Index

Scans `MONUSAC_ROOT` for all `.tif` / `.xml` pairs, loads and caches every sample
into memory-efficient numpy arrays, and derives multi-hot labels.

In [ ]:
def build_monusac_index(
    root: Path,
    target_size: int = IMG_SIZE,
    verbose: bool = True,
) -> pd.DataFrame:
    """Scan MONUSAC_ROOT and build a sample-level DataFrame.

    Columns:
        patient_id  : TCGA patient folder name
        sample_name : file stem (e.g. TCGA-55-1594-01Z-00-DX1_001)
        tif_path    : absolute path to .tif
        xml_path    : absolute path to .xml
        is_test     : bool — True if patient in OFFICIAL_TEST_PATIENTS
    """
    records = []
    patient_dirs = sorted([d for d in root.iterdir() if d.is_dir()])

    if verbose:
        print(f'Found {len(patient_dirs)} patient directories.')

    for pdir in patient_dirs:
        patient_id = pdir.name
        is_test    = patient_id in OFFICIAL_TEST_PATIENTS

        # Find all .tif files (excluding any that are .svs)
        tif_files = sorted(pdir.glob('*.tif'))
        for tif in tif_files:
            xml = tif.with_suffix('.xml')
            if not xml.exists():
                continue  # skip unannotated images
            records.append({
                'patient_id'  : patient_id,
                'sample_name' : tif.stem,
                'tif_path'    : str(tif),
                'xml_path'    : str(xml),
                'is_test'     : is_test,
            })

    df = pd.DataFrame(records)

    if verbose:
        print(f'Total samples    : {len(df)}')
        print(f'Test samples     : {df.is_test.sum()}')
        print(f'Train samples    : {(~df.is_test).sum()}')
        n_test_pts  = df[df.is_test]['patient_id'].nunique()
        n_train_pts = df[~df.is_test]['patient_id'].nunique()
        print(f'Test patients    : {n_test_pts}')
        print(f'Train patients   : {n_train_pts}')
        # Fallback check
        if n_test_pts == 0:
            print('WARNING: No official test patients found in directory names.')
            print('Falling back to random 80/20 patient split (seed=42).')

    return df


def load_all_samples(
    df: pd.DataFrame,
    target_size: int = IMG_SIZE,
) -> Tuple[np.ndarray, np.ndarray]:
    """Load all samples into memory. Returns (images, masks) arrays.

    images : (N, target_size, target_size, 3) uint8
    masks  : (N, target_size, target_size, 4) float32
    """
    N = len(df)
    images = np.zeros((N, target_size, target_size, 3), dtype=np.uint8)
    masks  = np.zeros((N, target_size, target_size, NUM_CLASSES), dtype=np.float32)

    for i, row in enumerate(tqdm(df.itertuples(), total=N, desc='Loading samples')):
        img, msk = load_monusac_sample(
            Path(row.tif_path), Path(row.xml_path), target_size
        )
        images[i] = img
        masks[i]  = msk

    return images, masks


# ── Build index ────────────────────────────────────────────────────────────────
print('Scanning MoNuSAC directory...')
df_index = build_monusac_index(MONUSAC_ROOT)

# ── Fallback: random patient-level 80/20 split if official test set not found ──
if df_index['is_test'].sum() == 0:
    rng_split = np.random.default_rng(SEED)
    all_patients = df_index['patient_id'].unique()
    rng_split.shuffle(all_patients)
    n_test_patients = max(1, int(len(all_patients) * 0.2))
    fallback_test   = set(all_patients[:n_test_patients])
    df_index['is_test'] = df_index['patient_id'].isin(fallback_test)
    print(f'Fallback split: {df_index.is_test.sum()} test / {(~df_index.is_test).sum()} train samples')

print(df_index.groupby('is_test').size().rename({True: 'Test', False: 'Train'}))

## 5. Load All Samples into Memory

In [ ]:
print('Loading all MoNuSAC images and masks into memory...')
all_images, all_masks = load_all_samples(df_index)

# ── Derive multi-hot labels ────────────────────────────────────────────────────
# label[i, c] = 1 if any nucleus of class c exists in image i
all_labels = (all_masks.sum(axis=(1, 2)) > 0).astype(np.uint8)  # (N, 4)

# ── Attach labels to index ─────────────────────────────────────────────────────
df_index['label_str'] = [
    '|'.join(CLASS_NAMES[c] for c in range(NUM_CLASSES) if all_labels[i, c])
    for i in range(len(df_index))
]
df_index['cardinality'] = all_labels.sum(axis=1)

print(f'\nImages shape : {all_images.shape}  dtype={all_images.dtype}')
print(f'Masks shape  : {all_masks.shape}  dtype={all_masks.dtype}')
print(f'Labels shape : {all_labels.shape}  dtype={all_labels.dtype}')
print(f'\nLabel cardinality distribution:')
print(pd.Series(all_labels.sum(axis=1)).value_counts().sort_index())
print(f'\nPer-class positive rate:')
for c, cname in enumerate(CLASS_NAMES):
    n_pos = all_labels[:, c].sum()
    print(f'  {cname:<14}: {n_pos}/{len(df_index)}  ({n_pos/len(df_index)*100:.1f}%)')

## 6. Train / Validation / Test Split

- **Test**: official MoNuSAC test patients (or fallback random 20% if not found)
- **Train**: remaining patients → 80% train, 20% val (patient-level)
- A fixed 50-image held-out test subset is additionally sampled for per-image CAM
  evaluation (seed=42), **identical in spirit to the PanNuke 50-image test set**.

In [ ]:
# ── Global indices ─────────────────────────────────────────────────────────────
test_mask_bool  = df_index['is_test'].values
train_mask_bool = ~test_mask_bool

test_global_indices  = np.where(test_mask_bool)[0]
trainval_global_idx  = np.where(train_mask_bool)[0]

# Patient-level 80/20 train/val from the non-test patients
rng_tv = np.random.default_rng(SEED)
train_patients = df_index.loc[train_mask_bool, 'patient_id'].unique()
rng_tv.shuffle(train_patients)
n_val_pts  = max(1, int(len(train_patients) * 0.2))
val_pts    = set(train_patients[:n_val_pts])
train_pts  = set(train_patients[n_val_pts:])

train_indices = np.where(
    df_index['patient_id'].isin(train_pts) & train_mask_bool
)[0]
val_indices   = np.where(
    df_index['patient_id'].isin(val_pts) & train_mask_bool
)[0]

# ── Sample exactly 50 test images (or all test images if < 50) ────────────────
rng_test = np.random.default_rng(SEED)
n_test_cam = min(50, len(test_global_indices))
test_cam_indices = rng_test.choice(
    test_global_indices, size=n_test_cam, replace=False
)
test_cam_indices = np.sort(test_cam_indices)

print(f'Train samples : {len(train_indices):,}')
print(f'Val samples   : {len(val_indices):,}')
print(f'Test (full)   : {len(test_global_indices):,}')
print(f'Test (CAM 50) : {len(test_cam_indices)} images (for per-image IoU evaluation)')
print(f'\nTest patient cardinality distribution:')
test_cards = all_labels[test_global_indices].sum(axis=1)
print(pd.Series(test_cards).value_counts().sort_index())

## 7. DataLoaders

In [ ]:
class MoNuSACDataset(Dataset):
    """Lightweight MoNuSAC dataset wrapping pre-loaded numpy arrays."""

    def __init__(
        self,
        images    : np.ndarray,   # (N, H, W, 3) uint8
        labels    : np.ndarray,   # (N, 4) uint8
        indices   : np.ndarray,   # subset indices into images/labels
        transform : T.Compose,
    ) -> None:
        self.images    = images
        self.labels    = labels
        self.indices   = indices
        self.transform = transform

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, i: int) -> Tuple[torch.Tensor, torch.Tensor]:
        g   = self.indices[i]
        img = self.images[g]               # (H, W, 3) uint8
        lbl = self.labels[g].astype(np.float32)  # (4,)
        return self.transform(img), torch.from_numpy(lbl)


train_tfm = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.05),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_tfm = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_ds = MoNuSACDataset(all_images, all_labels, train_indices, train_tfm)
val_ds   = MoNuSACDataset(all_images, all_labels, val_indices,   val_tfm)

# ── Compute class positive weights from training labels ────────────────────────
train_labels_arr = all_labels[train_indices].astype(np.float32)
pos_counts = train_labels_arr.sum(axis=0)                    # (4,)
neg_counts = len(train_labels_arr) - pos_counts
pos_weight = torch.tensor(
    neg_counts / (pos_counts + 1e-6), dtype=torch.float32
).to(DEVICE)
print(f'pos_weight: {pos_weight.cpu().numpy().round(2)}')

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')

## 8. DenseNet169 Multi-Label Model

**Identical architecture to the PanNuke experiment**, with `num_classes=4` instead of 5.
`Linear(1664→512) + BN + ReLU + Dropout(0.3) + Linear(512→4)`

In [ ]:
class DenseNet169MultiLabel(nn.Module):
    """DenseNet169 backbone + multi-label head.

    Identical to PanNuke experiment; only num_classes differs (4 vs 5).
    """

    def __init__(self, num_classes: int = NUM_CLASSES, dropout_p: float = 0.3) -> None:
        super().__init__()
        backbone = models.densenet169(weights=models.DenseNet169_Weights.IMAGENET1K_V1)
        in_features = backbone.classifier.in_features  # 1664
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.backbone(x))


model = DenseNet169MultiLabel(num_classes=NUM_CLASSES, dropout_p=DROPOUT_P).to(DEVICE)

total_p    = sum(p.numel() for p in model.parameters())
backbone_p = sum(p.numel() for p in model.backbone.parameters())
head_p     = sum(p.numel() for p in model.classifier.parameters())
print(f'Total parameters : {total_p/1e6:.2f} M')
print(f'Backbone         : {backbone_p/1e6:.2f} M')
print(f'Head             : {head_p/1e3:.1f} K')

## 9. Training

Identical training loop to PanNuke: AdamW + differential LR + CosineAnnealingLR + AMP + gradient clipping. Checkpoint saved at peak validation macro-AUC.

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(),    'lr': LR_BACKBONE},
    {'params': model.classifier.parameters(),  'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
scaler    = GradScaler()

history   = {'train_loss': [], 'val_loss': [], 'val_auc': []}
best_auc  = 0.0


def evaluate_epoch(
    mdl: nn.Module, loader: DataLoader
) -> Tuple[float, float]:
    mdl.eval()
    all_logits, all_labels_e = [], []
    total_loss = 0.0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            with autocast():
                logits = mdl(imgs)
                loss   = criterion(logits, lbls)
            total_loss += loss.item()
            all_logits.append(logits.cpu().float())
            all_labels_e.append(lbls.cpu().float())
    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels_e).numpy()
    probs_np  = torch.sigmoid(torch.tensor(logits_np)).numpy()
    auc_scores = []
    for c in range(NUM_CLASSES):
        if labels_np[:, c].sum() > 0 and (1 - labels_np[:, c]).sum() > 0:
            auc_scores.append(roc_auc_score(labels_np[:, c], probs_np[:, c]))
    macro_auc = float(np.mean(auc_scores)) if auc_scores else 0.0
    return total_loss / len(loader), macro_auc


print(f'Starting training ({NUM_EPOCHS} epochs)...')
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, lbls in tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}', leave=False):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, lbls)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    scheduler.step()
    train_loss /= len(train_loader)

    val_loss, val_auc = evaluate_epoch(model, val_loader)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)

    improved = val_auc > best_auc
    print(f'Epoch {epoch:02d} | train={train_loss:.4f} | val={val_loss:.4f} '
          f'| AUC={val_auc:.4f}' + (' ← best' if improved else ''))
    if improved:
        best_auc = val_auc
        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'val_auc': val_auc, 'val_loss': val_loss,
        }, MODEL_PATH)

print(f'\nTraining complete. Best val AUC: {best_auc:.4f}')
print(f'Checkpoint: {MODEL_PATH}')

## 10. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('DenseNet169 Training on MoNuSAC — Multi-Label Classification',
             fontsize=12, fontweight='bold')
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], 'b-o', markersize=4, label='Train loss')
axes[0].plot(epochs, history['val_loss'],   'r-s', markersize=4, label='Val loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['val_auc'], 'g-D', markersize=4, label='Val Macro AUC')
best_ep = int(np.argmax(history['val_auc'])) + 1
axes[1].axvline(best_ep, color='orange', linestyle='--', label=f'Best epoch {best_ep}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Macro AUC')
axes[1].set_title('Validation AUC'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig0_training_curves.png', bbox_inches='tight', dpi=300)
plt.show()
print('Training curves saved.')

## 11. Load Best Checkpoint & Test Set Inference

In [ ]:
ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded checkpoint: epoch {ckpt["epoch"]}, val AUC {ckpt["val_auc"]:.4f}')

_infer_tfm = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def preprocess(img_rgb: np.ndarray) -> torch.Tensor:
    """HWC uint8 → (1,3,H,W) float32 on DEVICE with requires_grad=True."""
    return _infer_tfm(img_rgb).unsqueeze(0).to(DEVICE).requires_grad_(True)


@torch.no_grad()
def get_pred(tensor: torch.Tensor) -> Tuple[np.ndarray, np.ndarray]:
    logits = model(tensor.detach())
    probs  = torch.sigmoid(logits).squeeze().cpu().numpy()
    return (probs >= THRESHOLD).astype(np.uint8), probs


# ── Full test set inference for classification metrics ────────────────────────
print('Running inference on full test set...')
gt_all, probs_all = [], []
for gidx in tqdm(test_global_indices, desc='Test inference'):
    t = preprocess(all_images[gidx])
    _, probs = get_pred(t)
    gt_all.append(all_labels[gidx])
    probs_all.append(probs)

gt_arr    = np.array(gt_all)     # (N_test, 4)
probs_arr = np.array(probs_all)  # (N_test, 4)
pred_arr  = (probs_arr >= THRESHOLD).astype(int)
print(f'Test set: {len(test_global_indices)} images')

## 12. Per-Label Classification Metrics

In [ ]:
per_class_metrics = {}
for ci, cname in enumerate(CLASS_NAMES):
    gt_c  = gt_arr[:, ci]
    pr_c  = pred_arr[:, ci]
    pb_c  = probs_arr[:, ci]
    has_both = gt_c.sum() > 0 and (1 - gt_c).sum() > 0
    per_class_metrics[cname] = {
        'AUC'      : round(float(roc_auc_score(gt_c, pb_c)), 4) if has_both else None,
        'Accuracy' : round(float(accuracy_score(gt_c, pr_c)), 4),
        'Precision': round(float(precision_score(gt_c, pr_c, zero_division=0)), 4),
        'Recall'   : round(float(recall_score(gt_c, pr_c,    zero_division=0)), 4),
        'F1'       : round(float(f1_score(gt_c, pr_c,        zero_division=0)), 4),
    }

valid_aucs = [v['AUC'] for v in per_class_metrics.values() if v['AUC'] is not None]
macro_auc  = round(float(np.mean(valid_aucs)), 4)
macro_f1   = round(float(np.mean([v['F1'] for v in per_class_metrics.values()])), 4)

metrics_out = {'per_class': per_class_metrics, 'macro_auc': macro_auc, 'macro_f1': macro_f1}
with open(OUT_DIR / 'monusac_per_label_metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)

print('Per-label metrics on full test set:')
print(f'{"Class":<16} {"AUC":>6} {"Acc":>6} {"Prec":>6} {"Rec":>6} {"F1":>6}')
print('-' * 48)
for cname, m in per_class_metrics.items():
    auc_s = f"{m['AUC']:.4f}" if m['AUC'] else '  N/A'
    print(f'{cname:<16} {auc_s:>6} {m["Accuracy"]:>6.4f} '
          f'{m["Precision"]:>6.4f} {m["Recall"]:>6.4f} {m["F1"]:>6.4f}')
print('-' * 48)
print(f'{"Macro":<16} {macro_auc:>6.4f} {"":>6} {"":>6} {"":>6} {macro_f1:>6.4f}')

## 13. CAM Engine — 2×2 Unified Implementation

**Identical to PanNuke** — same `CAMEngine` class, same FPN layer hooks, same
GradCAM/LayerCAM math. Only `model` reference changes.

In [ ]:
class CAMEngine:
    """Unified 2×2 CAM engine — identical to PanNuke implementation.

    Args:
        model        : trained DenseNet169MultiLabel
        use_pyramid  : False = single denseblock4; True = three-level FPN
        use_layercam : False = GradCAM (GAP); True = LayerCAM (pixel-wise)
        sharpen      : apply joint bilateral filter (pyramid only)
    """

    def __init__(
        self,
        model:        nn.Module,
        use_pyramid:  bool = False,
        use_layercam: bool = False,
        sharpen:      bool = True,
    ) -> None:
        self.model        = model
        self.use_pyramid  = use_pyramid
        self.use_layercam = use_layercam
        self.sharpen      = sharpen and use_pyramid
        self._handles: List = []
        self._layers: List[Tuple[str, float]] = (
            FPN_LAYERS if use_pyramid
            else [('backbone.features.denseblock4', 1.0)]
        )
        self._store: Dict[str, Dict] = {
            name: {'act': None, 'grad': None}
            for name, _ in self._layers
        }
        self._register_hooks()

    def _sub(self, dotpath: str) -> nn.Module:
        m = self.model
        for k in dotpath.split('.'):
            m = getattr(m, k)
        return m

    def _register_hooks(self) -> None:
        for name, _ in self._layers:
            mod = self._sub(name)
            n   = name
            self._handles.append(mod.register_forward_hook(
                lambda m, i, o, n=n: self._store[n].__setitem__('act', o.detach())
            ))
            self._handles.append(mod.register_full_backward_hook(
                lambda m, gi, go, n=n: self._store[n].__setitem__('grad', go[0].detach())
            ))

    def remove_hooks(self) -> None:
        for h in self._handles:
            h.remove()
        self._handles.clear()

    def _compute_level(
        self,
        act : torch.Tensor,
        grad: torch.Tensor,
        hw  : Tuple[int, int],
    ) -> np.ndarray:
        if self.use_layercam:
            cam = F.relu((F.relu(grad) * act).sum(dim=1, keepdim=True))
        else:
            alpha = grad.mean(dim=(2, 3), keepdim=True)
            cam   = F.relu((alpha * act).sum(dim=1, keepdim=True))
        arr    = cam.squeeze().cpu().numpy()
        arr    = np.maximum(arr, 0)
        interp = cv2.INTER_CUBIC if self.use_pyramid else cv2.INTER_LINEAR
        arr    = cv2.resize(arr, (hw[1], hw[0]), interpolation=interp)
        if arr.max() > 1e-8:
            arr /= arr.max()
        return arr.astype(np.float32)

    @staticmethod
    def _bilateral_sharpen(
        cam: np.ndarray, guide: np.ndarray
    ) -> np.ndarray:
        gray = cv2.cvtColor(guide, cv2.COLOR_RGB2GRAY)
        u8   = (cam * 255).astype(np.uint8)
        if hasattr(cv2, 'ximgproc') and hasattr(cv2.ximgproc, 'jointBilateralFilter'):
            s = cv2.ximgproc.jointBilateralFilter(gray, u8, 9, 75, 75)
        else:
            s = cv2.bilateralFilter(u8, 9, 75, 75)
        r = s.astype(np.float32) / 255.0
        if r.max() > 1e-8:
            r /= r.max()
        return r

    def generate(
        self,
        tensor   : torch.Tensor,
        class_idx: int,
        guide    : Optional[np.ndarray] = None,
    ) -> np.ndarray:
        H, W = tensor.shape[2], tensor.shape[3]
        self.model.zero_grad()
        self.model(tensor)[0, class_idx].backward(retain_graph=False)

        cams, weights = [], []
        for name, w in self._layers:
            a = self._store[name]['act']
            g = self._store[name]['grad']
            if a is not None and g is not None:
                cams.append(self._compute_level(a, g, (H, W)))
                weights.append(w)

        if not cams:
            return np.zeros((H, W), dtype=np.float32)

        total = sum(weights)
        fused = sum(w / total * c for w, c in zip(weights, cams))
        if fused.max() > 1e-8:
            fused /= fused.max()

        if self.sharpen and guide is not None:
            guide_r = cv2.resize(guide, (W, H))
            fused   = self._bilateral_sharpen(fused, guide_r)

        return fused.astype(np.float32)


def make_engines(mdl: nn.Module) -> Dict[str, CAMEngine]:
    return {
        'std_gradcam'  : CAMEngine(mdl, use_pyramid=False, use_layercam=False),
        'fpn_gradcam'  : CAMEngine(mdl, use_pyramid=True,  use_layercam=False),
        'std_layercam' : CAMEngine(mdl, use_pyramid=False, use_layercam=True),
        'fpn_layercam' : CAMEngine(mdl, use_pyramid=True,  use_layercam=True),
    }


print('CAMEngine defined. Instantiating four engines...')
engines = make_engines(model)
print('  ' + '  |  '.join(f"{k}: {METHOD_LABELS[k]}" for k in METHODS))

## 14. Shared Utilities

IoU, overlay, and triplet renderer — identical logic to PanNuke.

In [ ]:
def compute_iou(cam: np.ndarray, gt_ch: np.ndarray, thr: float) -> float:
    gt_bin = gt_ch > 0
    if not gt_bin.any():
        return float('nan')
    pred  = cam >= thr
    inter = (pred & gt_bin).sum()
    union = (pred | gt_bin).sum()
    return float(inter) / float(union) if union > 0 else float('nan')


def resize_ch(ch: np.ndarray, hw: Tuple[int, int]) -> np.ndarray:
    if ch.shape == hw:
        return ch
    return cv2.resize(ch, (hw[1], hw[0]), interpolation=cv2.INTER_NEAREST)


def render_gt_mask(
    mask_4ch: np.ndarray,
    hw: Tuple[int, int] = (IMG_SIZE, IMG_SIZE),
) -> np.ndarray:
    H, W   = hw
    canvas = np.zeros((H, W, 3), dtype=np.float32)
    oh, ow = mask_4ch.shape[:2]
    if (oh, ow) != (H, W):
        mask_4ch = np.stack([
            cv2.resize(mask_4ch[..., c], (W, H), interpolation=cv2.INTER_NEAREST)
            for c in range(NUM_CLASSES)
        ], axis=-1)
    colors_01 = [
        (int(c[1:3], 16) / 255, int(c[3:5], 16) / 255, int(c[5:7], 16) / 255)
        for c in CLASS_COLORS
    ]
    for c_idx, color in enumerate(colors_01):
        canvas[mask_4ch[..., c_idx] > 0] = color
    return (canvas * 255).clip(0, 255).astype(np.uint8)


def build_overlay(
    cam: np.ndarray, orig_resized: np.ndarray, color_hex: str,
    noise_thr: float = NOISE_THR, alpha_scale: float = ALPHA_SCALE,
) -> np.ndarray:
    r = int(color_hex[1:3], 16) / 255
    g = int(color_hex[3:5], 16) / 255
    b = int(color_hex[5:7], 16) / 255
    clean = np.where(cam >= noise_thr, cam, 0.0)
    rgba  = np.zeros((*cam.shape, 4), dtype=np.float32)
    rgba[..., 0] = clean * r
    rgba[..., 1] = clean * g
    rgba[..., 2] = clean * b
    rgba[..., 3] = clean * alpha_scale
    return rgba


def composite_overlay(
    cams: Dict[int, np.ndarray],
    active: List[int],
    orig_resized: np.ndarray,
) -> np.ndarray:
    H = W = IMG_SIZE
    comp = np.zeros((H, W, 4), dtype=np.float32)
    for c in active:
        layer    = build_overlay(cams[c], orig_resized, CLASS_COLORS[c])
        brighter = layer[..., 3] > comp[..., 3]
        comp[brighter] = layer[brighter]
    bg  = orig_resized.astype(np.float32) / 255.0
    a   = comp[..., 3:4]
    rgb = comp[..., :3]
    return ((rgb * a + bg * (1 - a)).clip(0, 1) * 255).astype(np.uint8)


def save_triplet(
    img_idx: int, orig: np.ndarray, gt_mask: np.ndarray,
    overlay: np.ndarray, active: List[int], probs: List[float],
    true_lbls: List[int], iou_at_50: Dict[int, float],
    method_lbl: str, save_path: Path,
) -> None:
    fig = plt.figure(figsize=(16, 5.6), facecolor='#0d0d0d')
    gs  = fig.add_gridspec(1, 3, wspace=0.04)
    titles = ['Original Image', 'Ground-Truth Mask', f'{method_lbl} Overlay']
    for col, (panel, title) in enumerate(zip([orig, gt_mask, overlay], titles)):
        ax = fig.add_subplot(gs[0, col])
        ax.imshow(panel)
        ax.set_title(title, color='white', fontsize=11, fontweight='bold', pad=5)
        ax.axis('off')
    patches = []
    for c in range(NUM_CLASSES):
        if c not in active:
            continue
        marker = '\u2713' if bool(true_lbls[c]) else '\u2717'
        iou_v  = iou_at_50.get(c, float('nan'))
        iou_s  = f'{iou_v:.3f}' if not np.isnan(iou_v) else 'N/A'
        r = int(CLASS_COLORS[c][1:3], 16) / 255
        g = int(CLASS_COLORS[c][3:5], 16) / 255
        b = int(CLASS_COLORS[c][5:7], 16) / 255
        patches.append(mpatches.Patch(
            facecolor=(r, g, b), edgecolor='white', linewidth=0.5,
            label=f'{marker} {CLASS_DISPLAY[c]}  p={probs[c]:.2f}  IoU={iou_s}',
        ))
    if patches:
        fig.axes[2].legend(
            handles=patches, loc='lower right', fontsize=7.4,
            framealpha=0.82, facecolor='#111111', edgecolor='#555555',
            labelcolor='white', handlelength=1.1, borderpad=0.6, labelspacing=0.45,
        )
    fig.suptitle(
        f'Image #{img_idx:03d}  |  MoNuSAC — {method_lbl}  |  \u2713=TP  \u2717=FP',
        color='#dddddd', fontsize=11, fontweight='bold', y=1.02,
    )
    plt.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)


print('Shared utilities defined.')

## 15. Section A — 50 CAM Test Images × 4 Methods

Builds `test_predictions.json` and `results_50_test.csv`.
Triplet figures saved to `triplets/{method}/`.

In [ ]:
for mname in METHODS:
    (TRIP_DIR / mname).mkdir(exist_ok=True)

test_records: List[Dict] = []
test_rows:    List[Dict] = []
N_TEST = len(test_cam_indices)

print(f'Processing {N_TEST} CAM test images × 4 methods...\n')

for local_i, gidx in enumerate(tqdm(test_cam_indices, desc='CAM test images')):
    orig_rgb  = all_images[gidx]       # (H, W, 3) uint8
    mask_4ch  = all_masks[gidx]        # (H, W, 4) float32
    gt_labels = all_labels[gidx]       # (4,) uint8

    # ── Prediction ─────────────────────────────────────────────────────────────
    t = preprocess(orig_rgb)
    pred_lbl, probs = get_pred(t)
    active = [i for i, p in enumerate(pred_lbl) if p == 1]

    orig_resized = cv2.resize(orig_rgb, (IMG_SIZE, IMG_SIZE))
    gt_mask_rgb  = render_gt_mask(mask_4ch)

    # ── Save test record ───────────────────────────────────────────────────────
    rec = {
        'image_idx'   : local_i,
        'global_idx'  : int(gidx),
        'patient_id'  : df_index.iloc[gidx]['patient_id'],
        'sample_name' : df_index.iloc[gidx]['sample_name'],
        'true_labels' : {cn: int(gt_labels[ci]) for ci, cn in enumerate(CLASS_NAMES)},
        'pred_labels' : {cn: int(pred_lbl[ci])  for ci, cn in enumerate(CLASS_NAMES)},
        'pred_probs'  : {cn: round(float(probs[ci]), 4) for ci, cn in enumerate(CLASS_NAMES)},
    }
    test_records.append(rec)

    row: Dict = {
        'image_idx'  : local_i,
        'global_idx' : int(gidx),
        'cardinality': int(gt_labels.sum()),
        'patient_id' : df_index.iloc[gidx]['patient_id'],
    }
    for ci, cname in enumerate(CLASS_NAMES):
        row[f'gt_{cname}']   = int(gt_labels[ci])
        row[f'pred_{cname}'] = int(pred_lbl[ci])
        row[f'prob_{cname}'] = round(float(probs[ci]), 4)

    # ── Generate all 4 CAMs ────────────────────────────────────────────────────
    all_cams: Dict[str, Dict[int, np.ndarray]] = {m: {} for m in METHODS}
    for c_idx in active:
        for mname, engine in engines.items():
            t_cam = preprocess(orig_rgb)
            cam   = engine.generate(t_cam, c_idx, guide=orig_resized)
            all_cams[mname][c_idx] = cam
            del t_cam
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    # ── Record IoU per class × method × threshold ──────────────────────────────
    for ci, cname in enumerate(CLASS_NAMES):
        gt_ch = resize_ch(mask_4ch[..., ci], (IMG_SIZE, IMG_SIZE))
        for mname in METHODS:
            cam = all_cams[mname].get(
                ci, np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
            )
            for thr in IOU_THRS:
                row[f'{mname}_iou_{thr:.2f}_{cname}'] = round(
                    compute_iou(cam, gt_ch, thr), 4)
            row[f'{mname}_cam_area_{cname}'] = round(
                float((cam >= NOISE_THR).mean()), 4)

    # ── Save triplet figures ───────────────────────────────────────────────────
    for mname in METHODS:
        overlay = (
            composite_overlay(all_cams[mname], active, orig_resized)
            if active else orig_resized.copy()
        )
        iou_50_vals = {
            c: compute_iou(
                all_cams[mname].get(c, np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)),
                resize_ch(mask_4ch[..., c], (IMG_SIZE, IMG_SIZE)), 0.50,
            ) for c in active
        }
        save_triplet(
            img_idx=local_i, orig=orig_resized, gt_mask=gt_mask_rgb,
            overlay=overlay, active=active, probs=list(probs),
            true_lbls=list(gt_labels), iou_at_50=iou_50_vals,
            method_lbl=METHOD_LABELS[mname],
            save_path=TRIP_DIR / mname / f'{mname}_{local_i:03d}.png',
        )

    test_rows.append(row)

# Save outputs
with open(TEST_JSON, 'w') as f:
    json.dump(test_records, f, indent=2)

df_test = pd.DataFrame(test_rows)
df_test.to_csv(OUT_DIR / 'results_50_test.csv', index=False)
print(f'\nCSV saved  : {OUT_DIR / "results_50_test.csv"}')
print(f'JSON saved : {TEST_JSON}')
print(f'Shape      : {df_test.shape}')

## 16. Figure 1 — IoU Comparison (50 Test Images)

In [ ]:
fig, axes = plt.subplots(1, len(IOU_THRS), figsize=(20, 6), sharey=False)
fig.suptitle('MoNuSAC — IoU Comparison (50 Test Images, 4 Methods × 3 Thresholds)',
             fontsize=13, fontweight='bold')
x = np.arange(NUM_CLASSES)
w = 0.18
offsets = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]

for ax, thr in zip(axes, IOU_THRS):
    for mname, off in zip(METHODS, offsets):
        means = [
            df_test[f'{mname}_iou_{thr:.2f}_{c}'].dropna().mean()
            for c in CLASS_NAMES
        ]
        bars = ax.bar(x + off, means, w,
                      label=METHOD_LABELS[mname],
                      color=METHOD_COLORS[mname],
                      edgecolor='white', linewidth=0.4, alpha=0.9)
        for b, v in zip(bars, means):
            if not np.isnan(v):
                ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.004,
                        f'{v:.3f}', ha='center', va='bottom',
                        fontsize=5.5, rotation=90)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_DISPLAY, rotation=20, ha='right', fontsize=9)
    ax.set_title(f'IoU @ {thr:.2f}', fontsize=10)
    ax.set_ylabel('Mean IoU')
    ax.set_ylim(0, 0.35)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_test_iou_4methods.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIG_DIR / 'fig1_test_iou_4methods.png', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 1 saved.')

## 17. Figure 2 — Macro IoU Summary Heatmap

In [ ]:
summary_rows = []
for mname in METHODS:
    for thr in IOU_THRS:
        class_means = [
            df_test[f'{mname}_iou_{thr:.2f}_{c}'].dropna().mean()
            for c in CLASS_NAMES
        ]
        summary_rows.append({
            'method'   : METHOD_LABELS[mname],
            'threshold': thr,
            'macro_iou': round(np.nanmean(class_means), 4),
            **{f'iou_{c}': round(v, 4) for c, v in zip(CLASS_DISPLAY, class_means)},
        })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_DIR / 'method_iou_summary.csv', index=False)
print('Macro IoU Summary (IoU@0.50):')
print(df_summary[df_summary['threshold'] == 0.50]
      [['method', 'macro_iou'] + [f'iou_{c}' for c in CLASS_DISPLAY]]
      .to_string(index=False))

# Factorial decomposition
df50 = df_summary[df_summary['threshold'] == 0.50]
def get_macro(m: str) -> float:
    return df50[df50['method'] == METHOD_LABELS[m]]['macro_iou'].values[0]
sg, fg = get_macro('std_gradcam'), get_macro('fpn_gradcam')
sl, fl = get_macro('std_layercam'), get_macro('fpn_layercam')
print(f'\n── Factorial decomposition (IoU@0.50) ──')
print(f'  SG={sg:.4f}  FG={fg:.4f}  SL={sl:.4f}  FL={fl:.4f}')
print(f'  LayerCAM main effect : {((sl-sg)+(fl-fg))/2:+.4f}')
print(f'  Pyramid main effect  : {((fg-sg)+(fl-sl))/2:+.4f}')
print(f'  Interaction term     : {(fl-fg)-(sl-sg):+.4f}')

# Heatmap
pivot_data = df_summary[df_summary['threshold'] == 0.50].copy()
hm_cols    = [f'iou_{c}' for c in CLASS_DISPLAY]
hm_mat     = pivot_data.set_index('method')[hm_cols].values.astype(float)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(hm_mat, ax=ax,
            xticklabels=CLASS_DISPLAY,
            yticklabels=pivot_data['method'].values,
            cmap='YlOrRd', annot=True, fmt='.4f', annot_kws={'size': 10},
            linewidths=0.5, linecolor='#cccccc',
            cbar_kws={'label': 'Mean IoU@0.50', 'shrink': 0.8})
ax.set_title('MoNuSAC — IoU@0.50 — 4 Methods × 4 Classes (50 test images)',
             fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=20, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_iou_heatmap_4methods.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIG_DIR / 'fig2_iou_heatmap_4methods.png', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 2 saved.')

## 18. Figure 3 — Factorial Axis Decomposition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    'MoNuSAC — Decomposing the 2×2 Design: Gradient Treatment vs Resolution Strategy\n'
    '(IoU@0.50, 50 test images)',
    fontsize=12, fontweight='bold',
)
x = np.arange(NUM_CLASSES)
w = 0.35

for ax, pairs, title in zip(axes, [
    [('std_gradcam','Standard GradCAM (GAP)'), ('std_layercam','Standard LayerCAM (pixel-wise)')],
    [('std_gradcam','Standard GradCAM (single)'), ('fpn_gradcam','FPN-GradCAM (pyramid)')],
], [
    'Gradient Treatment Axis\n(Single layer, denseblock4 only)',
    'Resolution Strategy Axis\n(GAP gradient weighting)',
]):
    for i, (mname, lbl) in enumerate(pairs):
        means = [df_test[f'{mname}_iou_0.50_{c}'].dropna().mean() for c in CLASS_NAMES]
        off   = (i - 0.5) * w
        bars  = ax.bar(x + off, means, w, label=lbl,
                       color=METHOD_COLORS[mname], edgecolor='white', linewidth=0.4, alpha=0.9)
        for b, v in zip(bars, means):
            if not np.isnan(v):
                ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.004,
                        f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_DISPLAY, rotation=18, ha='right', fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('Mean IoU@0.50')
    ax.set_ylim(0, 0.30)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_axis_decomposition.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIG_DIR / 'fig3_axis_decomposition.png', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 3 saved.')

## 19. Figure 4 — Visual Comparison Grid

In [ ]:
N_PREVIEW = 4
fig, axes = plt.subplots(N_PREVIEW, 5, figsize=(22, 5.6 * N_PREVIEW))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle(
    'MoNuSAC — Visual Comparison: GT Mask (col 1) + 4 Methods (cols 2–5)',
    color='white', fontsize=13, fontweight='bold', y=1.005,
)
col_titles = ['GT Mask'] + [METHOD_LABELS[m] for m in METHODS]

for row_i in range(N_PREVIEW):
    gidx = test_cam_indices[row_i]
    axes[row_i, 0].imshow(render_gt_mask(all_masks[gidx]))
    axes[row_i, 0].axis('off')
    if row_i == 0:
        axes[row_i, 0].set_title(col_titles[0], color='white', fontsize=10,
                                  fontweight='bold', pad=5)
    for col_i, mname in enumerate(METHODS):
        fpath = TRIP_DIR / mname / f'{mname}_{row_i:03d}.png'
        if fpath.exists():
            full_img = np.array(Image.open(fpath))
            pw = full_img.shape[1] // 3
            axes[row_i, col_i+1].imshow(full_img[:, 2*pw:])
        else:
            axes[row_i, col_i+1].text(0.5, 0.5, 'N/A', ha='center', color='white')
        axes[row_i, col_i+1].axis('off')
        if row_i == 0:
            axes[row_i, col_i+1].set_title(col_titles[col_i+1], color='white',
                                             fontsize=10, fontweight='bold', pad=5)

plt.tight_layout(pad=0.5)
plt.savefig(FIG_DIR / 'fig4_visual_comparison_grid.png',
            dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figure 4 saved.')

## 20. Section B — Full Dataset Inference (All 4 Methods)

Runs all 4 CAM methods on every correctly-classified image in the full dataset.

In [ ]:
full_rows: List[Dict] = []
skipped = 0
test_cam_set = set(test_cam_indices.tolist())

# Iterate over ALL images (train + val + test) for full interference analysis
print(f'Processing {len(all_images)} images × 4 methods (full dataset)...')

for gidx in tqdm(range(len(all_images)), desc='Full dataset'):
    gt_labels   = all_labels[gidx]
    cardinality = int(gt_labels.sum())
    img_raw     = all_images[gidx]
    mask_4ch    = all_masks[gidx]

    t_eval             = preprocess(img_raw)
    pred_labels, probs = get_pred(t_eval)

    if not np.array_equal(gt_labels, pred_labels):
        skipped += 1
        continue
    active = [i for i, p in enumerate(pred_labels) if p == 1]
    if not active:
        skipped += 1
        continue

    img_resized = cv2.resize(img_raw, (IMG_SIZE, IMG_SIZE))

    row: Dict = {
        'global_idx' : gidx,
        'patient_id' : df_index.iloc[gidx]['patient_id'],
        'cardinality': cardinality,
        'label_str'  : '|'.join(CLASS_NAMES[i] for i in range(NUM_CLASSES) if gt_labels[i]),
        'is_test'    : bool(df_index.iloc[gidx]['is_test']),
    }
    for ci, cname in enumerate(CLASS_NAMES):
        row[f'gt_{cname}']   = int(gt_labels[ci])
        row[f'pred_{cname}'] = int(pred_labels[ci])
        row[f'prob_{cname}'] = round(float(probs[ci]), 4)

    for c_idx in active:
        cname = CLASS_NAMES[c_idx]
        gt_ch = resize_ch(mask_4ch[..., c_idx], (IMG_SIZE, IMG_SIZE))
        for mname, engine in engines.items():
            t   = preprocess(img_raw)
            cam = engine.generate(t, c_idx, guide=img_resized)
            for thr in IOU_THRS:
                row[f'{mname}_iou_{thr:.2f}_{cname}'] = round(
                    compute_iou(cam, gt_ch, thr), 4)
            row[f'{mname}_cam_area_{cname}'] = round(
                float((cam >= NOISE_THR).mean()), 4)
            del t
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    full_rows.append(row)

for eng in engines.values():
    eng.remove_hooks()

df_full = pd.DataFrame(full_rows)
for mname in METHODS:
    for cname in CLASS_NAMES:
        for thr in IOU_THRS:
            col = f'{mname}_iou_{thr:.2f}_{cname}'
            if col not in df_full.columns:
                df_full[col] = np.nan

df_full.to_csv(OUT_DIR / 'results_full_dataset.csv', index=False)
print(f'\nRetained: {len(df_full)} images, skipped: {skipped}')
print('Cardinality distribution:')
print(df_full['cardinality'].value_counts().sort_index())

## 21. Figure 5 — IoU vs Label Cardinality

In [ ]:
def pooled_iou_by_card(df: pd.DataFrame, method: str, thr: float = 0.50) -> pd.DataFrame:
    rows = []
    for card in sorted(df['cardinality'].unique()):
        sub  = df[df['cardinality'] == card]
        vals = np.concatenate([
            sub[f'{method}_iou_{thr:.2f}_{c}'].dropna().values
            for c in CLASS_NAMES if f'{method}_iou_{thr:.2f}_{c}' in df.columns
        ])
        rows.append({
            'cardinality': card,
            'mean'       : float(np.nanmean(vals)),
            'sem'        : float(np.nanstd(vals) / max(np.sqrt(len(vals)), 1)),
            'n_obs'      : len(vals),
        })
    return pd.DataFrame(rows)


fig, ax = plt.subplots(figsize=(11, 6))
fig.suptitle(
    'MoNuSAC — IoU@0.50 vs Label Cardinality — All 4 Methods\n'
    '(correctly-classified images, all classes pooled)',
    fontsize=12, fontweight='bold',
)

spearman_results = {}
for mname in METHODS:
    cd = pooled_iou_by_card(df_full, mname)
    rho, p = stats.spearmanr(cd['cardinality'], cd['mean'])
    spearman_results[mname] = (rho, p)
    ax.errorbar(
        cd['cardinality'], cd['mean'],
        yerr=cd['sem'] * 1.96,
        fmt=f"{METHOD_MARKERS[mname]}-",
        color=METHOD_COLORS[mname],
        linewidth=2.5, markersize=9, capsize=5,
        label=f"{METHOD_LABELS[mname]} (\u03c1={rho:.3f})",
    )

ax.set_xlabel('Label Cardinality', fontsize=11)
ax.set_ylabel('Mean IoU@0.50 (95% CI)', fontsize=11)
ax.set_xticks(sorted(df_full['cardinality'].unique()))
ax.set_ylim(0, 0.25)
ax.legend(fontsize=10, loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_iou_vs_cardinality_4methods.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIG_DIR / 'fig5_iou_vs_cardinality_4methods.png', bbox_inches='tight', dpi=300)
plt.show()

print('Spearman monotonicity test:')
for mname, (rho, p) in spearman_results.items():
    print(f'  {METHOD_LABELS[mname]:<30}: rho={rho:.4f}, p={p:.4f}')
print('Figure 5 saved.')

## 22. Per-Class n=1 vs n≥2 + Figure 6

In [ ]:
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(20, 6), sharey=False)
fig.suptitle(
    'MoNuSAC — Per-Class IoU@0.50: n=1 (single) vs n≥2 (multi-label)',
    fontsize=12, fontweight='bold',
)

n1_nmulti_rows = []
for ax, cname, cdisp in zip(axes, CLASS_NAMES, CLASS_DISPLAY):
    single_means, multi_means = [], []
    for mname in METHODS:
        col    = f'{mname}_iou_0.50_{cname}'
        single = df_full[df_full['cardinality'] == 1][col].dropna().values
        multi  = df_full[df_full['cardinality'] >= 2][col].dropna().values
        sm     = float(np.nanmean(single)) if len(single) > 0 else np.nan
        mm     = float(np.nanmean(multi))  if len(multi)  > 0 else np.nan
        single_means.append(sm)
        multi_means.append(mm)

        if len(single) > 1 and len(multi) > 1:
            t_stat, p_val = stats.ttest_ind(single, multi, equal_var=False, nan_policy='omit')
        else:
            t_stat, p_val = np.nan, np.nan

        n1_nmulti_rows.append({
            'method'    : METHOD_LABELS[mname],
            'class'     : cdisp,
            'iou_n1'    : round(sm, 4) if not np.isnan(sm) else np.nan,
            'iou_nmulti': round(mm, 4) if not np.isnan(mm) else np.nan,
            'delta'     : round(sm - mm, 4) if not (np.isnan(sm) or np.isnan(mm)) else np.nan,
            'p_val'     : round(p_val, 4) if not np.isnan(p_val) else np.nan,
            'n_single'  : len(single),
            'n_multi'   : len(multi),
        })

    x_pos   = np.arange(len(METHODS))
    colors_ = [METHOD_COLORS[m] for m in METHODS]
    ax.bar(x_pos - 0.2, single_means, 0.35, label='n=1',  color=colors_, alpha=0.9,
           edgecolor='white', linewidth=0.4)
    ax.bar(x_pos + 0.2, multi_means,  0.35, label='n≥2',  color=colors_, alpha=0.55,
           edgecolor='white', linewidth=0.4, hatch='///')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([METHOD_LABELS[m].replace(' ', '\n') for m in METHODS], fontsize=6.5)
    ax.set_title(cdisp, fontsize=9, fontweight='bold')
    ax.set_ylabel('Mean IoU@0.50', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    if ax == axes[0]:
        from matplotlib.patches import Patch
        ax.legend(handles=[
            Patch(facecolor='grey', alpha=0.9,  label='n=1 (solid)'),
            Patch(facecolor='grey', alpha=0.55, hatch='///', label='n≥2 (hatched)'),
        ], fontsize=7, loc='upper right')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig6_per_class_n1_vs_nmulti.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIG_DIR / 'fig6_per_class_n1_vs_nmulti.png', bbox_inches='tight', dpi=300)
plt.show()

df_n1_nmulti = pd.DataFrame(n1_nmulti_rows)
df_n1_nmulti.to_csv(OUT_DIR / 'per_class_n1_vs_nmulti.csv', index=False)
print('Figure 6 + per_class_n1_vs_nmulti.csv saved.')

## 23. Pairwise Interference Analysis — All 4 Methods

In [ ]:
def build_interference(
    df: pd.DataFrame, method: str, thr: float = 0.50,
) -> Tuple[np.ndarray, pd.DataFrame]:
    thr_s  = f'{thr:.2f}'
    matrix = np.full((NUM_CLASSES, NUM_CLASSES), np.nan)
    rows   = []
    for c in range(NUM_CLASSES):
        s_c     = CLASS_NAMES[c]
        iou_col = f'{method}_iou_{thr_s}_{s_c}'
        if iou_col not in df.columns:
            continue
        base = df[(df[f'gt_{s_c}'] == 1) & (df[f'pred_{s_c}'] == 1)]
        for cp in range(NUM_CLASSES):
            if c == cp:
                continue
            s_cp    = CLASS_NAMES[cp]
            absent  = base[base[f'gt_{s_cp}'] == 0][iou_col].dropna().values
            present = base[base[f'gt_{s_cp}'] == 1][iou_col].dropna().values
            if len(absent) < 5 or len(present) < 5:
                continue
            delta = float(np.nanmean(absent)) - float(np.nanmean(present))
            t_stat, p_val = stats.ttest_ind(absent, present, equal_var=False, nan_policy='omit')
            matrix[c, cp] = delta
            rows.append({
                'method'     : method,
                'class_c'    : CLASS_DISPLAY[c],
                'class_cp'   : CLASS_DISPLAY[cp],
                'delta_iou'  : round(delta, 4),
                'iou_absent' : round(float(np.nanmean(absent)),  4),
                'iou_present': round(float(np.nanmean(present)), 4),
                'n_absent'   : len(absent),
                'n_present'  : len(present),
                't_stat'     : round(t_stat, 4),
                'p_val'      : round(p_val, 4),
                'significant': bool(p_val < 0.05),
                'direction'  : 'positive' if delta > 0 else 'negative',
            })
    return matrix, pd.DataFrame(rows)


int_matrices: Dict[str, np.ndarray] = {}
int_details:  List[pd.DataFrame]    = []

for mname in METHODS:
    mat, detail = build_interference(df_full, mname)
    int_matrices[mname] = mat
    int_details.append(detail)

df_pairwise_all = pd.concat(int_details, ignore_index=True)
df_pairwise_all.to_csv(OUT_DIR / 'pairwise_interference_all_methods.csv', index=False)
print('Pairwise interference CSV saved.')
print('Significant pairs per method:')
for mname in METHODS:
    sub = df_pairwise_all[
        (df_pairwise_all['method'] == mname) & df_pairwise_all['significant']
    ]
    print(f'  {METHOD_LABELS[mname]:<30}: {len(sub)} significant')

## 24. Figure 7 — 2×2 Interference Matrix Grid

In [ ]:
short = CLASS_DISPLAY  # already short enough
all_vals = np.concatenate([m.flatten() for m in int_matrices.values()])
vmax = np.nanmax(np.abs(all_vals))

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle(
    r'MoNuSAC — Pairwise Interference $\Delta$IoU$(c\mid c^{\prime})$ @ 0.50'
    '\n(positive = interference; row=measured class, col=co-occurring class)',
    fontsize=12, fontweight='bold',
)
positions = [(0,0,'std_gradcam'), (0,1,'fpn_gradcam'),
             (1,0,'std_layercam'), (1,1,'fpn_layercam')]

for r, c, mname in positions:
    ax  = axes[r][c]
    mat = int_matrices[mname]
    sns.heatmap(
        mat, ax=ax, mask=np.eye(NUM_CLASSES, dtype=bool),
        cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
        annot=True, fmt='.3f', annot_kws={'size': 10},
        xticklabels=short, yticklabels=short,
        linewidths=0.4, linecolor='#cccccc', cbar=(c == 1),
        cbar_kws={'label': '\u0394IoU', 'shrink': 0.85} if c == 1 else {},
    )
    ax.set_title(METHOD_LABELS[mname], fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=30, labelsize=9)
    ax.tick_params(axis='y', rotation=0,  labelsize=9)
    ax.set_xlabel("Co-occurring $c'$", fontsize=9)
    ax.set_ylabel('Measured $c$', fontsize=9)

fig.text(0.02, 0.73, 'Single Layer', ha='center', va='center',
         fontsize=11, fontweight='bold', color='#1F4E79', rotation=90)
fig.text(0.02, 0.27, 'Feature Pyramid', ha='center', va='center',
         fontsize=11, fontweight='bold', color='#1F4E79', rotation=90)
fig.text(0.28, 0.97, 'GAP Gradient (GradCAM)', ha='center', fontsize=11,
         fontweight='bold', color='#1F4E79')
fig.text(0.72, 0.97, 'Pixel-wise Gradient (LayerCAM)', ha='center', fontsize=11,
         fontweight='bold', color='#1F4E79')

plt.tight_layout(rect=[0.04, 0, 1, 0.96])
plt.savefig(FIG_DIR / 'fig7_interference_matrix_2x2.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIG_DIR / 'fig7_interference_matrix_2x2.png', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 7 saved.')

## 25. Permutation Tests — All 4 Methods

In [ ]:
def permutation_test(
    df: pd.DataFrame, method: str, c_idx: int, cp_idx: int,
    thr: float = 0.50, n_perm: int = N_PERMUTE, rng=None,
) -> Dict:
    if rng is None:
        rng = np.random.default_rng(SEED)
    s_c, s_cp = CLASS_NAMES[c_idx], CLASS_NAMES[cp_idx]
    iou_col   = f'{method}_iou_{thr:.2f}_{s_c}'
    if iou_col not in df.columns:
        return {}
    base = df[
        (df[f'gt_{s_c}'] == 1) & (df[f'pred_{s_c}'] == 1)
    ][[iou_col, f'gt_{s_cp}']].dropna()
    if len(base) < 10:
        return {}
    co_lbl = base[f'gt_{s_cp}'].values.copy()
    iou_v  = base[iou_col].values
    ab = iou_v[co_lbl == 0]
    pr = iou_v[co_lbl == 1]
    if len(ab) < 3 or len(pr) < 3:
        return {}
    obs  = float(np.nanmean(ab)) - float(np.nanmean(pr))
    null = np.array([
        np.nanmean(iou_v[rng.permutation(co_lbl) == 0])
        - np.nanmean(iou_v[rng.permutation(co_lbl) == 1])
        for _ in range(n_perm)
    ])
    p_emp = float((null >= obs).mean())
    return {
        'method'         : method,
        'class_c'        : CLASS_DISPLAY[c_idx],
        'class_cp'       : CLASS_DISPLAY[cp_idx],
        'observed_delta' : round(obs, 4),
        'null_mean'      : round(float(null.mean()), 4),
        'null_std'       : round(float(null.std()), 4),
        'p_empirical'    : round(p_emp, 4),
        'significant'    : bool(p_emp < 0.05),
        'n_absent'       : len(ab),
        'n_present'      : len(pr),
    }


rng_p = np.random.default_rng(SEED)
perm_rows = []
for mname in tqdm(METHODS, desc='Permutation tests'):
    for c in range(NUM_CLASSES):
        for cp in range(NUM_CLASSES):
            if c == cp:
                continue
            r = permutation_test(df_full, mname, c, cp, rng=rng_p)
            if r:
                perm_rows.append(r)

df_perm = pd.DataFrame(perm_rows)
df_perm.to_csv(OUT_DIR / 'permutation_results_all_methods.csv', index=False)
print('Permutation test results:')
for mname in METHODS:
    sub = df_perm[df_perm['method'] == mname]
    sig = sub['significant'].sum()
    print(f'  {METHOD_LABELS[mname]:<30}: {sig}/{len(sub)} significant')

## 26. Figure 8 — LayerCAM Robustness Comparison (Wilcoxon)

In [ ]:
pivot = df_pairwise_all.pivot_table(
    index=['class_c', 'class_cp'], columns='method', values='delta_iou',
).reset_index()
pivot.columns.name = None
pivot['gain_single'] = pivot.get('std_gradcam',  np.nan) - pivot.get('std_layercam', np.nan)
pivot['gain_fpn']    = pivot.get('fpn_gradcam',  np.nan) - pivot.get('fpn_layercam', np.nan)
pivot.to_csv(OUT_DIR / 'robustness_layercam_vs_gradcam.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(
    'MoNuSAC — LayerCAM Robustness Gain vs GradCAM\n'
    '(positive = LayerCAM less affected by interference)',
    fontsize=12, fontweight='bold',
)

for ax, (gain_col, mname_a, mname_b, title) in zip(axes, [
    ('gain_single', 'std_gradcam',  'std_layercam',
     'Single Layer:\nStd GradCAM \u2212 Std LayerCAM'),
    ('gain_fpn',    'fpn_gradcam',  'fpn_layercam',
     'Feature Pyramid:\nFPN-GradCAM \u2212 FPN-LayerCAM'),
]):
    sub   = pivot.dropna(subset=[gain_col]).sort_values(gain_col, ascending=True)
    if sub.empty:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', transform=ax.transAxes)
        continue
    labels  = [f"{r['class_c'][:10]}\n\u2192{r['class_cp'][:10]}" for _, r in sub.iterrows()]
    gains   = sub[gain_col].values
    colors_ = ['#4CAF50' if g > 0 else '#F44336' for g in gains]
    ax.barh(range(len(gains)), gains, color=colors_, alpha=0.85,
            edgecolor='white', linewidth=0.4)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('\u0394IoU reduction (positive = LayerCAM more robust)', fontsize=9)
    ax.grid(axis='x', alpha=0.3)

    valid = pivot.dropna(subset=[mname_a, mname_b])
    if len(valid) >= 5:
        w_stat, w_p = stats.wilcoxon(valid[mname_a], valid[mname_b], alternative='greater')
        ax.set_title(f'{title}\nWilcoxon W={w_stat:.0f}, p={w_p:.3f}',
                     fontsize=9, fontweight='bold')
        print(f'Wilcoxon ({mname_a} > {mname_b}): W={w_stat:.1f}, p={w_p:.4f}')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig8_layercam_robustness_gain.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIG_DIR / 'fig8_layercam_robustness_gain.png', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 8 saved.')

## 27. Figure 9 — Permutation Test: Null Distributions (Top Pair)

In [ ]:
if len(df_perm) > 0:
    top_pair = df_perm.loc[df_perm['observed_delta'].idxmax()]
    c_top  = CLASS_DISPLAY.index(top_pair['class_c'])
    cp_top = CLASS_DISPLAY.index(top_pair['class_cp'])

    fig, axes = plt.subplots(1, len(METHODS), figsize=(22, 5), sharey=False)
    fig.suptitle(
        f'MoNuSAC — Permutation Test — Top Interfering Pair: '
        f'{top_pair["class_c"]} | {top_pair["class_cp"]}\n'
        f'Null distribution (N={N_PERMUTE}) vs observed \u0394IoU for each method',
        fontsize=11, fontweight='bold',
    )

    rng2 = np.random.default_rng(SEED)
    for ax, mname in zip(axes, METHODS):
        s_c  = CLASS_NAMES[c_top]
        s_cp = CLASS_NAMES[cp_top]
        iou_col = f'{mname}_iou_0.50_{s_c}'
        if iou_col not in df_full.columns:
            ax.text(0.5, 0.5, 'N/A', ha='center', transform=ax.transAxes)
            continue
        base = df_full[
            (df_full[f'gt_{s_c}'] == 1) & (df_full[f'pred_{s_c}'] == 1)
        ][[iou_col, f'gt_{s_cp}']].dropna()
        if len(base) < 10:
            ax.text(0.5, 0.5, 'Insufficient data', ha='center', transform=ax.transAxes)
            continue
        co_lbl = base[f'gt_{s_cp}'].values
        iou_v  = base[iou_col].values
        obs    = np.nanmean(iou_v[co_lbl == 0]) - np.nanmean(iou_v[co_lbl == 1])
        null   = np.array([
            np.nanmean(iou_v[rng2.permutation(co_lbl) == 0])
            - np.nanmean(iou_v[rng2.permutation(co_lbl) == 1])
            for _ in range(N_PERMUTE)
        ])
        p_emp = float((null >= obs).mean())
        ax.hist(null, bins=40, color=METHOD_COLORS[mname], edgecolor='white', alpha=0.85)
        ax.axvline(obs, color='#E53935', linewidth=2.5, linestyle='--',
                   label=f'Observed={obs:.3f}')
        ax.axvline(null.mean(), color='#333333', linewidth=1.5, linestyle=':',
                   label=f'Null mean={null.mean():.3f}')
        ax.set_title(f'{METHOD_LABELS[mname]}\np={p_emp:.3f}', fontsize=9, fontweight='bold')
        ax.set_xlabel('\u0394IoU (permuted)', fontsize=9)
        ax.set_ylabel('Count', fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig9_permutation_all_methods.pdf', bbox_inches='tight', dpi=300)
    plt.savefig(FIG_DIR / 'fig9_permutation_all_methods.png', bbox_inches='tight', dpi=300)
    plt.show()
    print('Figure 9 saved.')
else:
    print('No permutation results — insufficient data for most pairs (small dataset).')

## 28. Figure 10 — Combined Publication Figure

In [ ]:
fig = plt.figure(figsize=(24, 16))
gs  = fig.add_gridspec(3, 4, hspace=0.45, wspace=0.38)
fig.suptitle(
    '2×2 Factorial CAM Study — MoNuSAC 2020 / DenseNet169\n'
    'Multi-Organ Nuclei Segmentation and Classification',
    fontsize=14, fontweight='bold', y=1.01,
)

# A: IoU heatmap
ax_a  = fig.add_subplot(gs[0, :2])
df50  = df_summary[df_summary['threshold'] == 0.50]
mat_a = np.array([
    [df50[df50['method'] == METHOD_LABELS[m]][f'iou_{c}'].values[0]
     if len(df50[df50['method'] == METHOD_LABELS[m]]) else np.nan
     for c in CLASS_DISPLAY]
    for m in METHODS
])
sns.heatmap(mat_a, ax=ax_a,
            xticklabels=CLASS_DISPLAY,
            yticklabels=[METHOD_LABELS[m] for m in METHODS],
            cmap='YlOrRd', annot=True, fmt='.3f', annot_kws={'size': 9},
            linewidths=0.4, linecolor='#cccccc',
            cbar_kws={'label': 'IoU@0.50', 'shrink': 0.8})
ax_a.set_title('(A) IoU@0.50 — Test Set', fontsize=10, fontweight='bold')
ax_a.tick_params(axis='x', rotation=25, labelsize=9)
ax_a.tick_params(axis='y', rotation=0,  labelsize=9)

# B: IoU vs cardinality
ax_b = fig.add_subplot(gs[0, 2:])
for mname in METHODS:
    cd = pooled_iou_by_card(df_full, mname)
    ax_b.plot(cd['cardinality'], cd['mean'],
              f"{METHOD_MARKERS[mname]}-",
              color=METHOD_COLORS[mname], linewidth=2, markersize=7,
              label=METHOD_LABELS[mname])
ax_b.set_xlabel('Label Cardinality', fontsize=9)
ax_b.set_ylabel('Mean IoU@0.50', fontsize=9)
ax_b.set_title('(B) IoU vs Co-occurrence Complexity', fontsize=10, fontweight='bold')
ax_b.legend(fontsize=8)
ax_b.grid(alpha=0.3)

# C-F: 2×2 interference matrices
for idx, (r, c, mname) in enumerate([
    (1, 0, 'std_gradcam'), (1, 1, 'fpn_gradcam'),
    (1, 2, 'std_layercam'), (1, 3, 'fpn_layercam'),
]):
    ax  = fig.add_subplot(gs[r, c])
    mat = int_matrices[mname]
    sns.heatmap(mat, ax=ax, mask=np.eye(NUM_CLASSES, dtype=bool),
                cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
                annot=True, fmt='.2f', annot_kws={'size': 8},
                xticklabels=[s[:8] for s in CLASS_DISPLAY],
                yticklabels=[s[:8] for s in CLASS_DISPLAY],
                linewidths=0.3, linecolor='#cccccc', cbar=False)
    letter = chr(67 + idx)
    ax.set_title(f'({letter}) {METHOD_LABELS[mname]}', fontsize=9, fontweight='bold')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)

# G: Single-layer robustness gain
ax_g = fig.add_subplot(gs[2, :2])
pv_s = pivot.dropna(subset=['std_gradcam', 'std_layercam'])
g_s  = pv_s['gain_single'].values if len(pv_s) > 0 else np.array([])
if len(g_s) > 0:
    ax_g.barh(range(len(g_s)), sorted(g_s),
              color=['#4CAF50' if v > 0 else '#F44336' for v in sorted(g_s)],
              alpha=0.85, edgecolor='white', linewidth=0.4)
ax_g.axvline(0, color='black', linewidth=1)
ax_g.set_xlabel('\u0394IoU reduction (Std GradCAM \u2212 Std LayerCAM)', fontsize=8)
ax_g.set_title('(G) Single-Layer: LayerCAM Robustness Gain', fontsize=10, fontweight='bold')
ax_g.grid(axis='x', alpha=0.3)

# H: FPN robustness gain
ax_h = fig.add_subplot(gs[2, 2:])
pv_f = pivot.dropna(subset=['fpn_gradcam', 'fpn_layercam'])
g_f  = pv_f['gain_fpn'].values if len(pv_f) > 0 else np.array([])
if len(g_f) > 0:
    ax_h.barh(range(len(g_f)), sorted(g_f),
              color=['#4CAF50' if v > 0 else '#F44336' for v in sorted(g_f)],
              alpha=0.85, edgecolor='white', linewidth=0.4)
ax_h.axvline(0, color='black', linewidth=1)
ax_h.set_xlabel('\u0394IoU reduction (FPN-GradCAM \u2212 FPN-LayerCAM)', fontsize=8)
ax_h.set_title('(H) FPN: LayerCAM Robustness Gain', fontsize=10, fontweight='bold')
ax_h.grid(axis='x', alpha=0.3)

plt.savefig(FIG_DIR / 'fig10_combined_main.pdf', bbox_inches='tight', dpi=300)
plt.savefig(FIG_DIR / 'fig10_combined_main.png', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 10 saved.')

## 29. Final Statistical Summary

In [ ]:
print('=' * 70)
print('FINAL STATISTICAL SUMMARY — MoNuSAC 2020 — ALL 4 CAM METHODS')
print('=' * 70)

print('\n(1) Classification metrics (full test set):')
print(f'    Macro AUC : {macro_auc:.4f}')
print(f'    Macro F1  : {macro_f1:.4f}')

print('\n(2) Macro IoU@0.50 on CAM test images:')
for mname in METHODS:
    macro = np.nanmean([
        df_test[f'{mname}_iou_0.50_{c}'].dropna().mean()
        for c in CLASS_NAMES
    ])
    print(f'    {METHOD_LABELS[mname]:<32}: {macro:.4f}')

print('\n(3) Factorial decomposition (IoU@0.50):')
print(f'    LayerCAM main effect : {((sl-sg)+(fl-fg))/2:+.4f}')
print(f'    Pyramid main effect  : {((fg-sg)+(fl-sl))/2:+.4f}')
print(f'    Interaction term     : {(fl-fg)-(sl-sg):+.4f}')

print('\n(4) Spearman rho (cardinality vs IoU@0.50):')
for mname in METHODS:
    cd = pooled_iou_by_card(df_full, mname)
    if len(cd) > 1:
        rho, p = stats.spearmanr(cd['cardinality'], cd['mean'])
        print(f'    {METHOD_LABELS[mname]:<32}: rho={rho:.4f}, p={p:.4f}')

print('\n(5) Significant interference pairs (permutation p<0.05):')
for mname in METHODS:
    if len(df_perm) > 0:
        sub = df_perm[df_perm['method'] == mname]
        sig = sub['significant'].sum()
        print(f'    {METHOD_LABELS[mname]:<32}: {sig}/{len(sub)}')

print('\n(6) Wilcoxon: LayerCAM interference vs GradCAM?')
for (ma, mb) in [('std_gradcam','std_layercam'), ('fpn_gradcam','fpn_layercam')]:
    valid = pivot.dropna(subset=[ma, mb])
    if len(valid) >= 5:
        w, p = stats.wilcoxon(valid[ma], valid[mb], alternative='greater')
        print(f'    H1 {METHOD_LABELS[ma]} > {METHOD_LABELS[mb]}: W={w:.0f}, p={p:.4f}')

print('\n(7) Negative interference pairs (significant, delta < 0):')
neg = df_pairwise_all[
    df_pairwise_all['significant'] & (df_pairwise_all['delta_iou'] < 0)
][['method', 'class_c', 'class_cp', 'delta_iou', 'p_val']]
if len(neg) > 0:
    print(neg.sort_values('delta_iou').to_string(index=False))
else:
    print('    None found.')

## 30. Output Directory Summary

In [ ]:
print('=' * 70)
print('MoNuSAC CAM COMPARISON — ALL OUTPUT FILES')
print('=' * 70)
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file():
        rel = str(f.relative_to(OUT_DIR))
        print(f'  {rel:<60}  {f.stat().st_size/1024:>7.1f} KB')

n_trip = sum(1 for _ in TRIP_DIR.rglob('*.png'))
print(f'\nTriplet figures : {n_trip} total')
print(f'  ({N_TEST} images × {len(METHODS)} methods = {N_TEST * len(METHODS)} figures)')
print(f'\nKey outputs for paper:')
print(f'  monusac_per_label_metrics.json  — compare with pannuke_densenet169 metrics')
print(f'  method_iou_summary.csv          — Table X in paper')
print(f'  pairwise_interference_*.csv     — interference comparison with PanNuke')
print(f'  permutation_results_*.csv       — causal interference evidence')
print(f'  robustness_layercam_vs_gradcam.csv — Wilcoxon cross-dataset comparison')
print(f'  figures/fig10_combined_main.png — main panel for paper')